<a href="https://colab.research.google.com/github/Rogerio-mack/IA_Classica2Agentes/blob/main/LLM_CSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LLM $\Rightarrow$ CSP, Esquema Geral**

```
Usuário
   |
LLM Parser
   |
JSON estruturado
   |
CSP Solver
   |
Plano válido
   |
LLM Explicador
   |
Resposta
```


## Esquema de Prompt

```
Você é um assistente responsável por converter descrições textuais de agendas corporativas em uma estrutura JSON para um sistema de otimização de reuniões.

Analise cuidadosamente o texto abaixo e extraia:

1. Funcionários disponíveis
2. Cargo/função de cada funcionário
3. Horários disponíveis de cada funcionário
4. Funcionários indisponíveis
5. Duração da reunião
6. Restrições gerais da empresa
7. Papéis mínimos necessários na reunião

Retorne APENAS um JSON válido, sem comentários adicionais.

Formato esperado:

{
"meeting_duration": int,
"required_roles": [],
"company_constraints": {
"lunch_break": [inicio, fim]
},
"employees": [
{
"name": "",
"role": "",
"available_slots": []
}
]
}

Texto:

"Ana e Maria são gerentes de vendas.
Maria está viajando hoje e não poderá participar.

Ana possui horários disponíveis das 9h às 12h e das 14h às 15h.

Bruno e João são engenheiros.
Bruno está disponível das 10h às 17h.
João está disponível apenas das 9h às 11h.

Carla e Fernanda trabalham na assessoria de comunicação.
Carla está disponível das 14h às 17h.
Fernanda está disponível das 9h às 10h e das 15h às 17h.

Precisamos marcar uma reunião de 2 horas envolvendo:

* um gerente,
* um engenheiro,
* e um assessor de comunicação.

Os funcionários da empresa almoçam diariamente das 12h às 14h.
```


## CSP or-tools

In [ ]:
from ortools.sat.python import cp_model

# ====================================
# "Saída do LLM"
# ====================================

# Imagine que isso veio do LLM

request = {
    "participants": ["Ana", "Bruno", "Carlos"],
    "duration": 2,
    "available_slots": {
        "Ana": [9,10,11,14,15],
        "Bruno": [10,11,14,15],
        "Carlos": [9,10,14,15]
    }
}

# ====================================
# CSP
# ====================================

model = cp_model.CpModel()

# Horário da reunião
meeting_time = model.NewIntVar(9, 15, "meeting_time")

# ====================================
# Restrições
# ====================================

for person, slots in request["available_slots"].items():

    model.AddAllowedAssignments(
        [meeting_time],
        [[s] for s in slots]
    )

# ====================================
# Resolver
# ====================================

solver = cp_model.CpSolver()

status = solver.Solve(model)

# ====================================
# Resultado
# ====================================

if status == cp_model.OPTIMAL:

    time = solver.Value(meeting_time)

    print(
        f"Reunião agendada às {time}:00"
    )

else:
    print("Sem solução.")

Reunião agendada às 10:00
